# Does the irregular grid's per-column construction create needlessly overlapping FOVs?

`test_irregular_grid_downstream_qc_tools.ipynb`'s Part 1 scatter plot
coloured FOVs by a synthetic, randomly-generated "intensity" value -- a
placeholder standing in for real per-FOV stats, purely to exercise the
heatmap-reshape code path. It was not real data and didn't mean anything
scientifically -- but looking at that plot surfaced a REAL, separate defect
in `build_irregular_bands` itself: within some columns (fixed `x`, adaptive
`y`), a few FOVs sit much closer together than the rest of that column's
otherwise near-`step_size_um` spacing.

**Root cause**: `build_irregular_bands` intersects each column's strip with
the tissue, and when a hole/notch splits that intersection into multiple
disjoint pieces, EACH piece gets its own independent `spaced_coords` call,
centred purely on that piece's own extent -- with no awareness of the
adjacent piece's phase. Where two pieces are separated by a hole narrower
than `step_size_um`, the last point of one piece and the first point of the
next can end up much closer together than `step_size_um` (real, measured
below), while every OTHER gap in the column (within one piece, always
exactly `step_size_um` by construction -- see `spaced_coords`' own
docstring) is fine.

This notebook (1) measures the real per-FOV overlap directly (replacing the
placeholder "intensity" with a real, meaningful metric), (2) identifies
which columns are affected and why, (3) prototypes and validates a fix
(redistribute each affected sub-column's own FOVs evenly between its own
end anchors, dropping any FOV the reclaimed span doesn't actually need at
the standard pitch), and (4) re-checks every invariant the prior two
notebooks already established still holds. Nothing is written to
`acquisition/positions.py` -- this stays a prototype, per the same
"test before promote" pattern as the rest of this line of work.

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from shapely.geometry import box as shapely_box
from shapely.ops import unary_union
from scipy.spatial import cKDTree
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components

MERCI_DIR  = Path(os.getcwd()).parent.parent.parent   # MERci/
SAMPLE_DIR = MERCI_DIR.parent
sys.path.insert(0, str(MERCI_DIR / "src"))

from MERci.acquisition.configs   import get_camera_pixel_size_um, get_camera_frame_size
from MERci.visualization import get_merci_figures_dir
from MERci.acquisition.positions import (
    load_boundary_polygon, load_hole_polygons,
    create_grid_positions, generate_scanning_path, filter_scanning_path, get_path_stats,
    spaced_coords,   # production internal -- reused, not reimplemented
)

FIG_DIR = get_merci_figures_dir(SAMPLE_DIR, "tests", "test_irregular_grid_column_overlap_correction", subfolder="irregular_grid")
FIG_DIR.mkdir(parents=True, exist_ok=True)

## Part 0 -- rebuild the real irregular grid (same boundary, same winning axis)

`build_irregular_bands`/`generate_irregular_scanning_path` duplicated
unchanged from the prior two notebooks (same convention they already
established). `fixed_axis="x"` again, matching
`compare_60x_40x_irregular_grid_fov_coverage.ipynb`'s own selected winner.
`bands` (the pre-path, per-column position lists) is kept around, not just
the final flat path -- the defect above lives at the per-column level, so
this notebook needs to inspect and repair it there.

In [ ]:
def build_irregular_bands(boundary_polygon, hole_polygons, step_size_um, fov_size_um,
                           fixed_axis="y", min_width_frac=0.1, fixed_offset=0.0):
    tissue = boundary_polygon.difference(unary_union(hole_polygons)) if hole_polygons else boundary_polygon
    xmin, ymin, xmax, ymax = boundary_polygon.bounds
    half_h = fov_size_um / 2.0
    min_width_um = min_width_frac * fov_size_um
    if fixed_axis == "y":
        fixed_min, fixed_max, cross_min, cross_max = ymin, ymax, xmin, xmax
    elif fixed_axis == "x":
        fixed_min, fixed_max, cross_min, cross_max = xmin, xmax, ymin, ymax
    else:
        raise ValueError("fixed_axis must be 'x' or 'y'")
    fixed_center = (fixed_min + fixed_max) / 2.0 + fixed_offset
    fixed_positions = spaced_coords(fixed_center, fixed_min, fixed_max, step_size_um, even=True)
    bands = []
    for f in fixed_positions:
        lo_f, hi_f = f - half_h, f + half_h
        strip = (shapely_box(cross_min - 1.0, lo_f, cross_max + 1.0, hi_f) if fixed_axis == "y"
                 else shapely_box(lo_f, cross_min - 1.0, hi_f, cross_max + 1.0))
        inter = tissue.intersection(strip)
        cross_vals = []
        if not inter.is_empty:
            pieces = list(inter.geoms) if hasattr(inter, "geoms") else [inter]
            pieces.sort(key=lambda p: p.bounds[0] if fixed_axis == "y" else p.bounds[1])
            for piece in pieces:
                if piece.is_empty:
                    continue
                pxmin, pymin, pxmax, pymax = piece.bounds
                lo, hi = (pxmin, pxmax) if fixed_axis == "y" else (pymin, pymax)
                if (hi - lo) < min_width_um:
                    continue
                piece_positions = spaced_coords((lo + hi) / 2.0, lo, hi, step_size_um, even=False)
                cross_vals.extend(piece_positions.tolist())
        bands.append((float(f), np.array(sorted(cross_vals))))
    return bands


def generate_irregular_scanning_path(bands, fixed_axis):
    path = []
    n_bands = len(bands)
    if fixed_axis == "y":
        for strip, i in enumerate(range(n_bands - 1, -1, -1)):
            fixed_val, cross_vals = bands[i]
            ordered = cross_vals if strip % 2 == 0 else cross_vals[::-1]
            for c in ordered:
                path.append((c, fixed_val))
    elif fixed_axis == "x":
        for j in range(n_bands):
            fixed_val, cross_vals = bands[j]
            ordered = cross_vals[::-1] if j % 2 == 0 else cross_vals
            for c in ordered:
                path.append((fixed_val, c))
    else:
        raise ValueError("fixed_axis must be 'x' or 'y'")
    return np.array(path) if path else np.empty((0, 2))


BOUNDARY_DIR = MERCI_DIR / "data" / "positions" / "examples" / "lineage_tracing_mosaic"
boundary_polygon = load_boundary_polygon(BOUNDARY_DIR / "boundary_positions.txt")
hole_polygons    = load_hole_polygons(BOUNDARY_DIR)

MICROSCOPE = "ST2"
image_size_px, _  = get_camera_frame_size(MICROSCOPE)
pixel_size_60x_um = get_camera_pixel_size_um(MICROSCOPE)
pixel_size_40x_um = pixel_size_60x_um * (60.0 / 40.0)
NON_OVERLAP_FRACTION = 0.9
fov_size_um  = pixel_size_40x_um * image_size_px
step_size_um = fov_size_um * NON_OVERLAP_FRACTION
FIXED_AXIS   = "x"
NOMINAL_OVERLAP_FRAC = 1.0 - NON_OVERLAP_FRACTION   # design overlap between adjacent FOVs, e.g. 0.10

bands_before = build_irregular_bands(boundary_polygon, hole_polygons, step_size_um, fov_size_um, fixed_axis=FIXED_AXIS)
path_before  = generate_irregular_scanning_path(bands_before, fixed_axis=FIXED_AXIS)
coords_before = filter_scanning_path(path_before, boundary_polygon, hole_polygons, fov_size_um)
print(f"fixed_axis='{FIXED_AXIS}': {len(coords_before)} FOVs before any fix, "
      f"nominal design overlap={NOMINAL_OVERLAP_FRAC:.2f}")

## Part 1 -- measure the real per-FOV overlap (replacing the meaningless placeholder)

Per-FOV overlap: for every FOV, the mean pairwise square-footprint overlap
fraction (`(fov_size_um - dx) * (fov_size_um - dy) / fov_area`, the same
formula the prior two notebooks already used for `connectivity_report`'s
overlap graph) against every OTHER FOV it actually overlaps. A FOV with
only its two expected in-column neighbours (each at the nominal
`~10%`-overlap spacing) should average close to `NOMINAL_OVERLAP_FRAC`; a
FOV caught in one of the artifacts described above will average much
higher.

In [ ]:
def per_fov_mean_overlap(coords, fov_size_um):
    n = len(coords)
    overlap_sum = np.zeros(n)
    overlap_n   = np.zeros(n, dtype=int)
    if n < 2:
        return overlap_sum
    tree = cKDTree(coords)
    pairs = tree.query_pairs(r=fov_size_um * np.sqrt(2))
    for i, j in pairs:
        dx = abs(coords[i, 0] - coords[j, 0])
        dy = abs(coords[i, 1] - coords[j, 1])
        if dx >= fov_size_um or dy >= fov_size_um:
            continue
        frac = (fov_size_um - dx) * (fov_size_um - dy) / (fov_size_um ** 2)
        overlap_sum[i] += frac; overlap_n[i] += 1
        overlap_sum[j] += frac; overlap_n[j] += 1
    mean_overlap = np.divide(overlap_sum, overlap_n, out=np.zeros(n), where=overlap_n > 0)
    return mean_overlap


def max_pairwise_overlap(coords, fov_size_um):
    """The single largest pairwise overlap fraction anywhere in the grid --
    more sensitive than the per-FOV MEAN above (which dilutes one bad
    neighbour against a FOV's other, normal neighbours)."""
    n = len(coords)
    if n < 2:
        return 0.0
    tree = cKDTree(coords)
    pairs = tree.query_pairs(r=fov_size_um * np.sqrt(2))
    best = 0.0
    for i, j in pairs:
        dx = abs(coords[i, 0] - coords[j, 0])
        dy = abs(coords[i, 1] - coords[j, 1])
        if dx >= fov_size_um or dy >= fov_size_um:
            continue
        frac = (fov_size_um - dx) * (fov_size_um - dy) / (fov_size_um ** 2)
        best = max(best, frac)
    return best


def plot_overlap_scatter(coords, mean_overlap, title, save_name):
    fig, ax = plt.subplots(figsize=(7, 8))
    sc = ax.scatter(coords[:, 0], coords[:, 1], c=mean_overlap, cmap="inferno", s=22,
                     vmin=0.0, vmax=max(0.5, np.percentile(mean_overlap, 99)))
    cbar = plt.colorbar(sc, ax=ax, fraction=0.035, pad=0.04)
    cbar.set_label("mean overlap fraction with overlapping neighbours")
    ax.axhline(0, color="none")   # no-op, keeps aspect logic simple
    ax.set_aspect("equal"); ax.invert_yaxis()
    ax.set_xlabel("Stage X (um)"); ax.set_ylabel("Stage Y (um)")
    ax.set_title(title, fontsize=11)
    fig.tight_layout()
    fig.savefig(FIG_DIR / save_name, dpi=150, bbox_inches="tight")
    plt.show()
    return fig


overlap_before = per_fov_mean_overlap(coords_before, fov_size_um)
plot_overlap_scatter(coords_before, overlap_before,
                      f"BEFORE fix -- {len(coords_before)} FOVs, coloured by mean overlap fraction\n"
                      f"(nominal design overlap = {NOMINAL_OVERLAP_FRAC:.2f})",
                      "test_irregular_grid_column_overlap_before.png")

n_flagged = int((overlap_before > 3 * NOMINAL_OVERLAP_FRAC).sum())
max_pair_before = max_pairwise_overlap(coords_before, fov_size_um)
print(f"max PER-FOV MEAN overlap fraction: {overlap_before.max():.3f} (nominal={NOMINAL_OVERLAP_FRAC:.2f})")
print(f"max single PAIRWISE overlap fraction: {max_pair_before:.3f} -- more sensitive, a bad neighbour "
      f"is diluted by a FOV's other normal neighbours in the per-FOV mean above.")
print(f"FOVs with mean overlap > 3x nominal: {n_flagged} / {len(coords_before)}")

## Part 2 -- identify the affected columns

Working directly on `bands_before` (before path-ordering/filtering, where
the defect actually originates): for each column, a gap between
consecutive cross-axis positions is "bad" if it implies noticeably more
overlap than by design (`overlap_frac > BAD_OVERLAP_FRAC`, well above
`NOMINAL_OVERLAP_FRAC` -- chosen generously above the nominal ~0.10 so only
genuine artifacts are flagged, not ordinary rounding). A TRUE zero-overlap
gap (`overlap_frac == 0`, i.e. the two FOVs' footprints don't touch at all)
is a real, deliberate non-adjacency (a genuine hole/gap) and is never
flagged.

In [ ]:
BAD_OVERLAP_FRAC = 0.3   # well above the nominal ~0.10 design overlap -- an explicit, stated threshold

def column_gap_report(cross_vals, fov_size_um, bad_overlap_frac=BAD_OVERLAP_FRAC):
    if len(cross_vals) < 2:
        return np.array([]), np.array([], dtype=bool)
    gaps = np.diff(cross_vals)
    overlap_frac = np.clip((fov_size_um - gaps) / fov_size_um, 0.0, None)
    return overlap_frac, overlap_frac > bad_overlap_frac

affected_rows = []
for fixed_val, cross_vals in bands_before:
    overlap_frac, is_bad = column_gap_report(cross_vals, fov_size_um)
    if is_bad.any():
        affected_rows.append({"x": fixed_val, "n_pts": len(cross_vals),
                               "n_bad_gaps": int(is_bad.sum()),
                               "worst_overlap_frac": float(overlap_frac.max())})

df_affected = pd.DataFrame(affected_rows).sort_values("worst_overlap_frac", ascending=False)
print(f"{len(df_affected)} / {len(bands_before)} columns have at least one bad (overlap_frac > "
      f"{BAD_OVERLAP_FRAC}) gap, {df_affected['n_bad_gaps'].sum()} bad gaps total.")
df_affected

## Part 3 -- the fix: redistribute each affected sub-column evenly

A **sub-column** is a maximal run of consecutive points bounded by either
the column's own ends or a TRUE zero-overlap gap (a real hole/gap that must
never be redistributed across). If a sub-column contains at least one bad
gap, ALL of its points (not just the two immediately touching the bad gap
-- a 2-point cluster has no interior to redistribute into, tried first and
confirmed broken) are re-spaced evenly between the sub-column's own first
and last position, via `linspace` -- spreading the excess overlap across
every gap in that sub-column instead of leaving it concentrated in one
place. If the reclaimed span doesn't actually need that many FOVs at the
standard `step_size_um` pitch, the surplus is dropped (the same
`spaced_coords`-style count formula, applied to the sub-column's own
span) -- directly answering "could 1 or 2 FOVs be removed here."

In [ ]:
def fix_column_overlap_clusters(cross_vals, fov_size_um, step_size_um, bad_overlap_frac=BAD_OVERLAP_FRAC):
    """Returns (new_cross_vals, applied) where applied is a list of
    (lo_idx, hi_idx, n_before, n_after) for every sub-column that was
    actually redistributed (n_before == n_after when only spacing changed,
    n_after < n_before when surplus FOVs were also dropped).
    """
    cross_vals = np.array(sorted(cross_vals), dtype=float)
    n = len(cross_vals)
    if n < 2:
        return cross_vals.copy(), []

    gaps = np.diff(cross_vals)
    overlap_frac = np.clip((fov_size_um - gaps) / fov_size_um, 0.0, None)
    disjoint = overlap_frac <= 0.0   # a TRUE gap -- never redistribute across this
    is_bad   = overlap_frac > bad_overlap_frac

    sub_ranges = []
    start = 0
    for i, d in enumerate(disjoint):
        if d:
            sub_ranges.append((start, i))
            start = i + 1
    sub_ranges.append((start, n - 1))

    fixed, applied = [], []
    for lo, hi in sub_ranges:
        seg_bad = is_bad[lo:hi] if hi > lo else np.array([], dtype=bool)
        if hi == lo or not seg_bad.any():
            fixed.extend(cross_vals[lo:hi + 1].tolist())
            continue
        span = cross_vals[hi] - cross_vals[lo]
        n_before = hi - lo + 1
        expected_n = max(2, int(round(span / step_size_um)) + 1)
        n_after = min(n_before, expected_n)
        fixed.extend(np.linspace(cross_vals[lo], cross_vals[hi], n_after).tolist())
        applied.append((lo, hi, n_before, n_after))
    return np.array(sorted(fixed)), applied


bands_after, fix_log = [], []
for fixed_val, cross_vals in bands_before:
    fixed_cv, applied = fix_column_overlap_clusters(cross_vals, fov_size_um, step_size_um)
    bands_after.append((fixed_val, fixed_cv))
    for lo, hi, n_before, n_after in applied:
        fix_log.append({"x": fixed_val, "lo_idx": lo, "hi_idx": hi,
                         "n_before": n_before, "n_after": n_after, "n_removed": n_before - n_after})

df_fix_log = pd.DataFrame(fix_log)
print(f"{len(df_fix_log)} sub-columns redistributed across {df_fix_log['x'].nunique()} columns, "
      f"{df_fix_log['n_removed'].sum()} FOVs removed as surplus.")
df_fix_log

In [ ]:
path_after   = generate_irregular_scanning_path(bands_after, fixed_axis=FIXED_AXIS)
coords_after = filter_scanning_path(path_after, boundary_polygon, hole_polygons, fov_size_um)

overlap_after = per_fov_mean_overlap(coords_after, fov_size_um)
plot_overlap_scatter(coords_after, overlap_after,
                      f"AFTER fix -- {len(coords_after)} FOVs, coloured by mean overlap fraction\n"
                      f"(nominal design overlap = {NOMINAL_OVERLAP_FRAC:.2f})",
                      "test_irregular_grid_column_overlap_after.png")

max_pair_after = max_pairwise_overlap(coords_after, fov_size_um)
print(f"n_fovs:                  before={len(coords_before)}, after={len(coords_after)}")
print(f"max PER-FOV MEAN overlap: before={overlap_before.max():.3f}, after={overlap_after.max():.3f} "
      f"(nominal={NOMINAL_OVERLAP_FRAC:.2f})")
print(f"max single PAIRWISE overlap: before={max_pair_before:.3f}, after={max_pair_after:.3f}")
print(f"FOVs > 3x nominal overlap: before={int((overlap_before > 3*NOMINAL_OVERLAP_FRAC).sum())}, "
      f"after={int((overlap_after > 3*NOMINAL_OVERLAP_FRAC).sum())}")
assert max_pair_after < max_pair_before, "the fix should reduce the worst-case single-pair overlap, not just move it."
assert len(coords_after) <= len(coords_before), "the fix should only ever remove FOVs, never add them."
print("\nPASS: the fix measurably reduces worst-case overlap (both metrics) and never increases FOV count.")

### Zoomed-in view of the single worst affected column, before vs. after

The full-grid scatter above makes the worst offenders visible as bright
outlier points, but a zoomed panel on the single worst column (by
pre-fix `worst_overlap_frac`, from Part 2's table) makes the actual
FOV-footprint clumping -- and the fix -- directly visible, the same way the
original report visually spotted this.

In [ ]:
worst_x = float(df_affected.iloc[0]["x"])

def plot_column_zoom(coords, x_val, fov_size_um, title, ax):
    mask = np.isclose(coords[:, 0], x_val, atol=1e-6)
    col_coords = coords[mask]
    half = fov_size_um / 2.0
    for x, y in col_coords:
        ax.add_patch(mpatches.Rectangle((x - half, y - half), fov_size_um, fov_size_um,
                                         lw=1.0, edgecolor="black", facecolor="steelblue", alpha=0.35))
    ax.scatter(col_coords[:, 0], col_coords[:, 1], s=12, color="black", zorder=3)
    ax.set_title(f"{title} ({len(col_coords)} FOVs)", fontsize=10)
    ax.set_aspect("equal"); ax.invert_yaxis()
    ax.set_xlim(x_val - fov_size_um, x_val + fov_size_um)

fig, axes = plt.subplots(1, 2, figsize=(8, 10), sharey=True)
plot_column_zoom(coords_before, worst_x, fov_size_um, "before fix", axes[0])
plot_column_zoom(coords_after,  worst_x, fov_size_um, "after fix",  axes[1])
fig.suptitle(f"column x={worst_x:.1f} -- worst pre-fix overlap", fontsize=11)
fig.tight_layout()
fig.savefig(FIG_DIR / "test_irregular_grid_column_overlap_zoom.png", dpi=150, bbox_inches="tight")
plt.show()

## Part 4 -- re-check every invariant the prior two notebooks established

The fix only moves/drops points WITHIN a column, after `build_irregular_
bands` -- it shouldn't be able to break the boustrophedon-path regression
match, single-connectivity, or measured tissue coverage. Checked directly,
not assumed.

In [ ]:
# (a) degenerate-rectangle regression: a plain rectangle has no holes, so no
# sub-column can ever contain a bad gap -- the fix must be a total no-op.
STEP_TEST, FOV_TEST = 100.0, 90.0
rect = shapely_box(0.0, 0.0, 733.0, 517.0)
rect_bands = build_irregular_bands(rect, [], STEP_TEST, FOV_TEST, fixed_axis=FIXED_AXIS)
rect_bands_fixed, rect_fix_log = [], []
for fixed_val, cross_vals in rect_bands:
    fixed_cv, applied = fix_column_overlap_clusters(cross_vals, FOV_TEST, STEP_TEST)
    rect_bands_fixed.append((fixed_val, fixed_cv))
    rect_fix_log.extend(applied)
assert not rect_fix_log, "the fix should never trigger on a plain rectangle (no holes -> no bad gaps)."
rect_path_before = generate_irregular_scanning_path(rect_bands, fixed_axis=FIXED_AXIS)
rect_path_after  = generate_irregular_scanning_path(rect_bands_fixed, fixed_axis=FIXED_AXIS)
assert np.allclose(rect_path_before, rect_path_after)
print("PASS (a): the fix is a total no-op on a plain rectangular tissue (no holes).")

In [ ]:
# (b) single connected component, before and after
def n_components(coords, fov_size_um, min_overlap_fraction=0.02):
    n = len(coords)
    if n == 0:
        return 0
    tree = cKDTree(coords)
    pairs = tree.query_pairs(r=fov_size_um * np.sqrt(2))
    ii, jj, frac = [], [], []
    fov_area = fov_size_um ** 2
    for i, j in pairs:
        dx = abs(coords[i, 0] - coords[j, 0]); dy = abs(coords[i, 1] - coords[j, 1])
        if dx >= fov_size_um or dy >= fov_size_um:
            continue
        f = (fov_size_um - dx) * (fov_size_um - dy) / fov_area
        if f >= min_overlap_fraction:
            ii.append(i); jj.append(j); frac.append(f)
    adj = coo_matrix((np.ones(len(frac)), (ii, jj)), shape=(n, n))
    n_comp, _ = connected_components(adj, directed=False)
    return n_comp

nc_before = n_components(coords_before, fov_size_um)
nc_after  = n_components(coords_after, fov_size_um)
print(f"connected components: before={nc_before}, after={nc_after}")
assert nc_before == 1, "the pre-fix grid was already expected to be a single connected component."
assert nc_after == 1, "the fix must not break single-connectivity."
print("PASS (b): single connected component preserved by the fix.")

In [ ]:
# (c) tissue coverage: union of FOV footprints (intersected with real tissue) / tissue area
def coverage_fraction(coords, fov_size_um, boundary_polygon, hole_polygons):
    tissue = boundary_polygon.difference(unary_union(hole_polygons)) if hole_polygons else boundary_polygon
    half = fov_size_um / 2.0
    squares = [shapely_box(x - half, y - half, x + half, y + half) for x, y in coords]
    covered = unary_union(squares).intersection(tissue)
    return covered.area / tissue.area

cov_before = coverage_fraction(coords_before, fov_size_um, boundary_polygon, hole_polygons)
cov_after  = coverage_fraction(coords_after,  fov_size_um, boundary_polygon, hole_polygons)
print(f"coverage_fraction: before={cov_before:.4f}, after={cov_after:.4f}")
assert cov_after >= cov_before - 1e-6, (
    f"the fix dropped real tissue coverage ({cov_before:.4f} -> {cov_after:.4f}) -- "
    f"a dropped 'surplus' FOV must never have been the only FOV covering some real tissue.")
print("PASS (c): tissue coverage is not reduced by the fix.")

## Takeaways

- **Real defect, not a visualisation artifact**: `build_irregular_bands`
  places each column's tissue PIECES (split by holes/notches) on
  independently-phased lattices with no awareness of each other -- where a
  hole is narrower than `step_size_um`, the two pieces' nearest boundary
  points can end up far closer together than intended (measured: gaps as
  small as ~1-6 um on the real benchmark, vs. the nominal ~273 um pitch --
  see Part 2's table). Every OTHER gap in the same column stays exactly
  `step_size_um` by construction (`spaced_coords`), so the effect is
  exactly the localised "some FOVs close together, the rest regular"
  pattern originally spotted in the placeholder heatmap's scatter fallback.
- **A per-pair fix doesn't work -- confirmed by trying it first**: fixing
  only the two points immediately touching a bad gap is a no-op whenever
  the bad gap sits between exactly two points (linspace of 2 fixed
  endpoints reproduces them exactly). The working fix instead identifies
  whole **sub-columns** (runs bounded by a TRUE zero-overlap gap -- a real
  hole -- or the column's own ends) and redistributes ALL of that
  sub-column's points evenly, spreading the excess overlap across every
  gap instead of leaving it concentrated in one place.
- **Measured effect on the real benchmark**: 18 of 36 columns had at least
  one bad gap (39 bad gaps total -- Part 2); the fix redistributed 21
  sub-columns across those same 18 columns (Part 3). The worst SINGLE
  pairwise overlap fraction (the more sensitive metric -- a per-FOV MEAN
  dilutes one bad neighbour against a FOV's other, normal ones) dropped
  from **0.997 to 0.206**, and the worst per-FOV MEAN dropped from 0.266 to
  0.117 (nominal design is 0.10 -- i.e. the worst case now sits barely
  above nominal, not effectively total overlap). FOV count dropped from
  572 to 550 (22 surplus FOVs removed -- the reclaimed sub-column spans
  didn't need that many at the standard `step_size_um` pitch), landing
  almost EXACTLY at the regular grid's own 550-FOV baseline. This directly
  confirms the "1 or 2 FOVs removable per column" suspicion, aggregated
  across every affected column.
- **All three prior invariants re-verified, not assumed**: the fix is a
  total no-op on a plain rectangular tissue (Part 4a), never breaks
  single-connectivity (Part 4b), and does not reduce measured tissue
  coverage (Part 4c).
- **Net conclusion**: this sub-column redistribution is not an optional
  nicety -- it is a REQUIRED post-processing step for any production
  version of the single-axis-adaptive irregular grid (Method 2), applied
  right after `build_irregular_bands` and before
  `generate_irregular_scanning_path`. The still-pending promotion plan
  for `acquisition/positions.py` should include this fix from the start,
  not as a follow-up patch.